# Planificador de palabras clave para SEO · DUPAVI

Hecho por [Eduard Pampalona](https://dupavi.es/recursos/planificador-palabras-clave/), consultor SEO · [dupavi.es](https://dupavi.es/)

Escribes una palabra clave y te devuelve las búsquedas relacionadas que hace la gente en Google, con su volumen, la tendencia, el CPC, la dificultad y la intención, ya agrupadas por tema.

**Qué necesitas**
- Una cuenta de [DataForSEO](https://dataforseo.com/) con saldo. Cada búsqueda cuesta céntimos: de 3 a 17 céntimos de dólar por palabra y país, según cuántas ideas traiga, con la intención incluida (tarifa de noviembre de 2026). Al acabar, la herramienta te dice lo que ha costado y el saldo que te queda.
- Las claves de la API de DataForSEO: el login y la contraseña que salen en tu panel, en «API Access». Ojo: no son tu correo ni la contraseña con la que entras en la web.

**Cómo se usa**
1. Guarda tus claves en los Secretos de Colab: el icono de la llave, a la izquierda. Crea `DATAFORSEO_LOGIN` y `DATAFORSEO_PASSWORD` y activa «Acceso al cuaderno». Así no quedan escritas en el cuaderno.
2. Ejecuta el paso 1 con el botón ▶. Solo una vez cada vez que abras el cuaderno.
3. Rellena el paso 2 y ejecútalo.
4. Si quieres, filtra en el paso 3 (no gasta saldo) y descárgalo en el paso 4.

**Lo que no hace**
- No decide por ti. Te da las búsquedas ordenadas; cuáles vale la pena trabajar depende de tu negocio, de tu web y de quién se lleva hoy esas búsquedas.
- Los volúmenes son estimaciones (datos de Google Ads, a través de DataForSEO). La intención y los grupos se calculan de forma automática: revísalos antes de decidir nada.
- En los temas que Google no deja anunciar (por ejemplo, el CBD), Google no da el volumen de algunas búsquedas. Eso no quiere decir que nadie las busque. No salen en la tabla, pero no se pierden: las tienes en la hoja «Descartadas» del Excel, con el motivo.


In [ ]:
#@title 1. Prepara la herramienta { display-mode: "form" }
#@markdown Ejecútala una vez cada vez que abras el cuaderno. Comprueba tus claves de DataForSEO y te dice el saldo que te queda. No gasta nada.

import re
import unicodedata
import warnings
from datetime import date

import numpy as np
import pandas as pd
import requests
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from google.colab import data_table, files, userdata
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

API = "https://api.dataforseo.com/v3"

# Tope de búsquedas en cada ejecución (palabras × países), para no gastar de más sin querer.
MAX_BUSQUEDAS = 20
# Cuanto más alto, más grandes los grupos (entre 0 y 1).
UMBRAL_GRUPOS = 0.7
# Con más palabras clave que esto, las de menos volumen se quedan sin agrupar.
MAX_AGRUPAR = 4000

IDIOMAS = {
    "Español": "es", "Catalán": "ca", "Inglés": "en", "Francés": "fr", "Alemán": "de",
    "Italiano": "it", "Portugués": "pt", "Neerlandés": "nl", "Sueco": "sv", "Ruso": "ru",
    "Turco": "tr", "Árabe": "ar", "Hindi": "hi", "Japonés": "ja", "Coreano": "ko",
    "Chino (simplificado)": "zh-cn",
}

STOPWORDS_NLTK = {
    "es": "spanish", "ca": "catalan", "en": "english", "fr": "french", "de": "german",
    "it": "italian", "pt": "portuguese", "nl": "dutch", "sv": "swedish", "ru": "russian",
    "tr": "turkish", "ar": "arabic",
}

INTENCIONES = {
    "informational": "Informativa", "navigational": "Navegacional",
    "commercial": "Comercial", "transactional": "Transaccional",
}

# Nombres de países en español → como los llama DataForSEO.
PAISES_ES = {
    "espana": "spain", "mexico": "mexico", "mejico": "mexico", "argentina": "argentina",
    "colombia": "colombia", "peru": "peru", "chile": "chile", "venezuela": "venezuela",
    "ecuador": "ecuador", "guatemala": "guatemala", "bolivia": "bolivia", "honduras": "honduras",
    "paraguay": "paraguay", "uruguay": "uruguay", "el salvador": "el salvador",
    "nicaragua": "nicaragua", "costa rica": "costa rica", "panama": "panama",
    "republica dominicana": "dominican republic", "puerto rico": "puerto rico",
    "estados unidos": "united states", "eeuu": "united states", "ee. uu.": "united states",
    "usa": "united states", "reino unido": "united kingdom", "inglaterra": "united kingdom",
    "gran bretana": "united kingdom", "irlanda": "ireland", "alemania": "germany",
    "francia": "france", "italia": "italy", "portugal": "portugal", "brasil": "brazil",
    "paises bajos": "netherlands", "holanda": "netherlands", "belgica": "belgium",
    "suiza": "switzerland", "austria": "austria", "suecia": "sweden", "noruega": "norway",
    "dinamarca": "denmark", "finlandia": "finland", "polonia": "poland", "grecia": "greece",
    "canada": "canada", "australia": "australia", "japon": "japan", "china": "china",
    "india": "india", "corea del sur": "south korea", "turquia": "turkey", "marruecos": "morocco",
}

# Para enseñar en español los países más habituales.
NOMBRE_PAIS = {
    "Spain": "España", "Mexico": "México", "Argentina": "Argentina", "Colombia": "Colombia",
    "Peru": "Perú", "Chile": "Chile", "Venezuela": "Venezuela", "Ecuador": "Ecuador",
    "Guatemala": "Guatemala", "Bolivia": "Bolivia", "Uruguay": "Uruguay", "Paraguay": "Paraguay",
    "Costa Rica": "Costa Rica", "Panama": "Panamá", "Dominican Republic": "República Dominicana",
    "United States": "Estados Unidos", "United Kingdom": "Reino Unido", "Ireland": "Irlanda",
    "Germany": "Alemania", "France": "Francia", "Italy": "Italia", "Portugal": "Portugal",
    "Brazil": "Brasil", "Netherlands": "Países Bajos", "Belgium": "Bélgica", "Switzerland": "Suiza",
    "Austria": "Austria", "Canada": "Canadá", "Australia": "Australia",
}

# Ciudades que Google nombra distinto de como se buscan en España.
ALIAS_CIUDADES = {
    "seville": ["sevilla"], "saragossa": ["zaragoza"], "a coruna": ["coruna", "la coruna"],
    "castellon de la plana": ["castellon"], "lleida": ["lerida"], "girona": ["gerona"],
    "ourense": ["orense"], "palma": ["palma de mallorca"],
}

SESION = None
GASTO = {"total": 0.0}
RESULTADOS = None
FILTRADO = None
OTRAS_CIUDADES = None
DESCARTADAS = None
BUSQUEDA = {}
_PAISES = None


def normalizar(texto):
    texto = unicodedata.normalize("NFD", str(texto).lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", texto).strip()


def dolares(cantidad):
    return f"{cantidad:.3f}".replace(".", ",") + " $"


def _llamar(metodo, ruta, tarea=None):
    """Llama a la API y devuelve el JSON, o None si algo falla (y explica qué)."""
    url = f"{API}/{ruta}"
    try:
        if metodo == "GET":
            r = SESION.get(url, timeout=120)
        else:
            r = SESION.post(url, json=[tarea], timeout=180)
    except requests.RequestException as error:
        print(f"❌ No he podido conectar con DataForSEO: {error}")
        return None
    if r.status_code == 401:
        print("❌ DataForSEO no acepta tus claves. Revisa el login y la contraseña de «API Access».")
        return None
    try:
        datos = r.json()
    except ValueError:
        print(f"❌ DataForSEO ha respondido con un error {r.status_code}.")
        return None
    GASTO["total"] += float(datos.get("cost") or 0)
    codigo = str(datos.get("status_code", ""))
    if codigo.startswith("401"):
        print("❌ DataForSEO no acepta tus claves. Revisa el login y la contraseña de «API Access».")
        return None
    if codigo.startswith("402"):
        print("❌ No te queda saldo en DataForSEO.")
        return None
    if codigo != "20000":
        print(f"❌ DataForSEO: {datos.get('status_message')}")
        return None
    return datos


def _resultado(datos):
    """El «result» de la primera tarea, o None si la tarea ha fallado."""
    if not datos or not datos.get("tasks"):
        return None
    tarea = datos["tasks"][0]
    if str(tarea.get("status_code")) != "20000":
        print(f"⚠️ DataForSEO: {tarea.get('status_message')}")
        return None
    return tarea.get("result") or []


def _saldo():
    """El saldo que queda. Si DataForSEO no lo da en ese momento (tiene un límite por minuto), no avisa: no hace falta."""
    datos = _llamar("GET", "appendix/user_data")
    try:
        return datos["tasks"][0]["result"][0]["money"]["balance"]
    except (TypeError, KeyError, IndexError):
        return None


def conectar():
    global SESION
    login = password = ""
    if EN_COLAB:
        try:
            login = userdata.get("DATAFORSEO_LOGIN") or ""
            password = userdata.get("DATAFORSEO_PASSWORD") or ""
        except Exception:
            login = password = ""
    if not (login and password):
        from getpass import getpass
        print("No encuentro tus claves en los Secretos de Colab (el icono de la llave, a la izquierda).")
        print("Puedes escribirlas ahora, solo para esta sesión: no se guardan en el cuaderno.")
        login = input("Login de la API de DataForSEO: ").strip()
        password = getpass("Contraseña de la API: ").strip()
    SESION = requests.Session()
    SESION.auth = (login, password)
    saldo = _saldo()
    if saldo is None:
        SESION = None
        print("No se ha podido conectar. Revisa las claves y vuelve a ejecutar el paso 1.")
        return False
    print(f"✅ Conectado a DataForSEO. Saldo: {dolares(saldo)}")
    return True


def _nombre(pais):
    return NOMBRE_PAIS.get(pais["location_name"], pais["location_name"])


def _cargar_paises():
    global _PAISES
    if _PAISES is None:
        res = _resultado(_llamar("GET", "dataforseo_labs/locations_and_languages"))
        if res is None:
            return []
        _PAISES = [p for p in res if p.get("location_type") == "Country"]
    return _PAISES


def buscar_pais(nombre):
    termino = normalizar(nombre)
    termino = PAISES_ES.get(termino, termino)
    paises = _cargar_paises()
    for p in paises:
        if normalizar(p.get("location_name", "")) == termino:
            return p
    if len(termino) == 2:
        for p in paises:
            if normalizar(p.get("country_iso_code", "")) == termino:
                return p
    parecidos = [p for p in paises if termino and termino in normalizar(p.get("location_name", ""))]
    if len(parecidos) == 1:
        return parecidos[0]
    if parecidos:
        opciones = ", ".join(p["location_name"] for p in parecidos[:8])
        print(f"⚠️ «{nombre}» puede ser: {opciones}. Escríbelo tal cual.")
    else:
        print(f"⚠️ No encuentro el país «{nombre}».")
    return None


def _idioma_del_pais(pais, codigo):
    disponibles = pais.get("available_languages") or []
    if not disponibles:
        return codigo
    for idioma in disponibles:
        if (idioma.get("language_code") or "").lower() == codigo.lower():
            return idioma["language_code"]
    nombres = ", ".join(sorted({i.get("language_name", "") for i in disponibles}))
    print(f"⚠️ En {_nombre(pais)} no hay datos en ese idioma. Idiomas disponibles: {nombres}.")
    return None


def _ideas(semilla, pais, idioma, profundidad):
    tarea = {
        "keyword": semilla, "location_code": pais["location_code"], "language_code": idioma,
        "depth": int(profundidad), "limit": 1000, "include_seed_keyword": True,
    }
    res = _resultado(_llamar("POST", "dataforseo_labs/google/related_keywords/live", tarea))
    if not res:
        return []
    r0 = res[0] or {}
    datos = []
    semilla_datos = r0.get("seed_keyword_data")
    for s in (semilla_datos if isinstance(semilla_datos, list) else [semilla_datos]):
        if s:
            datos.append(s.get("keyword_data", s))
    for item in r0.get("items") or []:
        if item.get("keyword_data"):
            datos.append(item["keyword_data"])
    return datos


def _fila(kd, pais, semilla):
    info = kd.get("keyword_info") or {}
    props = kd.get("keyword_properties") or {}
    intencion = kd.get("search_intent_info") or {}
    tendencia = info.get("search_volume_trend") or {}
    fila = {
        "País": _nombre(pais),
        "Semilla": semilla,
        "Palabra clave": kd.get("keyword") or "",
        "Volumen": info.get("search_volume"),
        "Intención": INTENCIONES.get(intencion.get("main_intent"), ""),
        "Otras intenciones": ", ".join(INTENCIONES.get(i, i) for i in (intencion.get("foreign_intent") or [])),
        "Dificultad": props.get("keyword_difficulty"),
        "CPC": info.get("cpc"),
        "Competencia": info.get("competition"),
        "Tendencia 3 meses (%)": tendencia.get("quarterly"),
        "Tendencia 12 meses (%)": tendencia.get("yearly"),
    }
    for mes in info.get("monthly_searches") or []:
        if mes.get("year") and mes.get("month"):
            fila[f"Vol {mes['year']}-{int(mes['month']):02d}"] = mes.get("search_volume")
    return fila


def _intenciones(df, idioma):
    """La intención de cada búsqueda, con su propio análisis (unos céntimos más).
    La que trae la búsqueda de ideas falla más; solo se usa si este análisis no responde."""
    claves = df["Palabra clave"].drop_duplicates().tolist()
    principal, otras = {}, {}
    for i in range(0, len(claves), 1000):
        tarea = {"keywords": claves[i:i + 1000], "language_code": idioma}
        res = _resultado(_llamar("POST", "dataforseo_labs/google/search_intent/live", tarea))
        for item in ((res or [{}])[0] or {}).get("items") or []:
            etiqueta = (item.get("keyword_intent") or {}).get("label")
            if etiqueta:
                principal[item.get("keyword")] = INTENCIONES.get(etiqueta, etiqueta)
                otras[item.get("keyword")] = ", ".join(
                    INTENCIONES.get(s.get("label"), s.get("label"))
                    for s in item.get("secondary_keyword_intents") or [] if s.get("label"))
    if not principal:
        print("⚠️ No he podido pedir la intención aparte: uso la que trae la búsqueda de ideas.")
        return df
    hay = df["Palabra clave"].isin(principal)
    df.loc[hay, "Intención"] = df.loc[hay, "Palabra clave"].map(principal)
    df.loc[hay, "Otras intenciones"] = df.loc[hay, "Palabra clave"].map(otras)
    return df


def _variantes(nombre):
    n = normalizar(nombre)
    variantes = {n}
    for separador in ("-", "/"):
        if separador in n:
            variantes.update(parte.strip() for parte in n.split(separador))
    for v in list(variantes):
        variantes.update(ALIAS_CIUDADES.get(v, []))
    return {v for v in variantes if v}


_CIUDADES = {}


def _ciudades(paises):
    """Nombres de las ciudades de esos países (la lista es gratis y se guarda para la siguiente búsqueda)."""
    nombres = set()
    for p in paises:
        iso = (p.get("country_iso_code") or "").lower()
        if not iso:
            continue
        if iso not in _CIUDADES:
            lugares = _resultado(_llamar("GET", f"serp/google/locations/{iso}"))
            if lugares is None:
                continue
            _CIUDADES[iso] = set()
            for lugar in lugares:
                if lugar.get("location_type") in ("City", "Municipality", "Province"):
                    _CIUDADES[iso].update(_variantes(lugar["location_name"].split(",")[0]))
        nombres |= _CIUDADES[iso]
    return {n for n in nombres if len(n) >= 4}


def _lugares(paises):
    """Ciudades y países: al agrupar no cuentan, para que «precio madrid» no junte temas distintos."""
    lugares = set(_ciudades(paises))
    lugares.update(PAISES_ES)
    lugares.update(PAISES_ES.values())
    lugares.update(normalizar(p.get("location_name", "")) for p in paises)
    return {l for l in lugares if len(l) >= 4}


def _sin_lugares(texto, lugares):
    palabras = normalizar(texto).split()
    quedan, i = [], 0
    while i < len(palabras):
        for n in range(min(4, len(palabras) - i), 0, -1):
            if " ".join(palabras[i:i + n]) in lugares:
                i += n
                break
        else:
            quedan.append(palabras[i])
            i += 1
    return " ".join(quedan)


def _menciona(texto, conjunto, max_palabras=4):
    palabras = texto.split()
    for n in range(1, max_palabras + 1):
        for i in range(len(palabras) - n + 1):
            trozo = " ".join(palabras[i:i + n])
            if trozo in conjunto:
                return trozo
    return None


def _stopwords(codigo):
    nombre = STOPWORDS_NLTK.get(codigo.split("-")[0].lower())
    if not nombre:
        return []
    try:
        import nltk
        try:
            palabras = nltk.corpus.stopwords.words(nombre)
        except LookupError:
            nltk.download("stopwords", quiet=True)
            palabras = nltk.corpus.stopwords.words(nombre)
        return sorted({normalizar(p) for p in palabras})
    except Exception:
        return []


def agrupar(df, idioma, lugares=frozenset()):
    """Agrupa por tema según las palabras que comparten (sin contar ciudades ni países)."""
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.feature_extraction.text import TfidfVectorizer

    volumen = df.groupby("Palabra clave")["Volumen"].max().sort_values(ascending=False)
    textos = list(volumen.index)
    grupo = {t: "Otros" for t in textos}
    agrupables = textos[:MAX_AGRUPAR]
    for t in textos[MAX_AGRUPAR:]:
        grupo[t] = "Sin agrupar (demasiadas)"
    if len(agrupables) >= 2:
        vectorizador = TfidfVectorizer(
            preprocessor=lambda t: _sin_lugares(t, lugares), stop_words=_stopwords(idioma) or None,
            sublinear_tf=True,
        )
        try:
            X = vectorizador.fit_transform(agrupables)
        except ValueError:
            X = None
        if X is not None:
            con_texto = np.where(X.getnnz(axis=1) > 0)[0]
            if len(con_texto) >= 2:
                Xs = X[con_texto]
                similitud = (Xs @ Xs.T).toarray()
                etiquetas = AgglomerativeClustering(
                    n_clusters=None, metric="precomputed", linkage="average",
                    distance_threshold=UMBRAL_GRUPOS,
                ).fit_predict(np.clip(1 - similitud, 0, 1))
                miembros = {}
                for posicion, etiqueta in enumerate(etiquetas):
                    miembros.setdefault(etiqueta, []).append(posicion)
                for posiciones in miembros.values():
                    if len(posiciones) < 2:
                        continue
                    # El nombre del grupo: la búsqueda que más se parece a las demás (si empatan, la de más volumen).
                    parecido = similitud[np.ix_(posiciones, posiciones)].sum(axis=1)
                    lista = [agrupables[con_texto[p]] for p in posiciones]
                    cabeza = max(zip(parecido.round(6), [volumen[t] for t in lista], lista))[2]
                    for t in lista:
                        grupo[t] = cabeza
    df = df.copy()
    df["Grupo"] = df["Palabra clave"].map(grupo)
    return df


def _intencion_principal(trozo):
    con = trozo[trozo["Intención"] != ""]
    if con.empty:
        return ""
    return con.groupby("Intención")["Volumen"].sum().idxmax()


def _media(serie, decimales):
    valor = pd.to_numeric(serie, errors="coerce").astype("float64").mean()
    return None if pd.isna(valor) else round(valor, decimales)


def resumen_grupos(df):
    filas = []
    for nombre, trozo in df.groupby("Grupo"):
        filas.append({
            "Grupo": nombre,
            "Palabras clave": trozo["Palabra clave"].nunique(),
            "Volumen total": int(trozo["Volumen"].sum()),
            "Intención principal": _intencion_principal(trozo),
            "Dificultad media": _media(trozo["Dificultad"], 0),
            "CPC medio": _media(trozo["CPC"], 2),
        })
    resumen = pd.DataFrame(filas)
    if resumen.empty:
        return resumen
    resumen["_al_final"] = resumen["Grupo"].isin(["Otros", "Sin agrupar (demasiadas)"])
    resumen = resumen.sort_values(["_al_final", "Volumen total"], ascending=[True, False])
    return resumen.drop(columns="_al_final").reset_index(drop=True)


def _ordenar(df):
    orden = {g: i for i, g in enumerate(resumen_grupos(df)["Grupo"])}
    df = df.assign(_orden=df["Grupo"].map(orden))
    df = df.sort_values(["_orden", "Volumen"], ascending=[True, False]).drop(columns="_orden")
    meses = sorted(c for c in df.columns if c.startswith("Vol "))
    primeras = ["Grupo", "Palabra clave", "Volumen", "Intención", "Otras intenciones", "Dificultad", "CPC",
                "Competencia", "Tendencia 3 meses (%)", "Tendencia 12 meses (%)", "País", "Semilla"]
    return df[[c for c in primeras if c in df.columns] + meses].reset_index(drop=True)


def mostrar(df, filas=20):
    df = df.copy()
    for columna in df.columns:
        if str(df[columna].dtype) == "Int64":
            df[columna] = df[columna].astype("float64")
    if EN_COLAB:
        display(data_table.DataTable(df, include_index=False, num_rows_per_page=filas))
    else:
        display(df.head(filas))


def _columnas_vista(df):
    columnas = ["Grupo", "Palabra clave", "Volumen", "Intención", "Dificultad", "CPC", "Tendencia 12 meses (%)"]
    if df["País"].nunique() > 1:
        columnas.append("País")
    return df[[c for c in columnas if c in df.columns]]


SIN_DATO = "Google no da el volumen"


def _apartar(tabla, quitar, motivo, apartadas):
    """Quita esas filas de la tabla y las guarda con su motivo, para la hoja «Descartadas»."""
    if quitar.any():
        apartadas.append(tabla[quitar].assign(Motivo=motivo))
    return tabla[~quitar]


def _tabla_descartadas(apartadas):
    if not apartadas:
        return None
    tabla = pd.concat(apartadas, ignore_index=True)
    # Si Google no da el volumen, la celda se queda vacía: no es un 0.
    tabla["Volumen"] = tabla["Volumen"].where(tabla["Motivo"] != SIN_DATO).astype("Int64")
    tabla = tabla.sort_values(["Motivo", "Volumen", "Palabra clave"], ascending=[True, False, True])
    return tabla[["Palabra clave", "Motivo", "Volumen", "País", "Semilla"]].reset_index(drop=True)


def buscar(palabras_clave, paises, idioma, profundidad, ciudad="", quitar_sueltas=True, volumen_minimo=0):
    global RESULTADOS, FILTRADO, OTRAS_CIUDADES, DESCARTADAS, BUSQUEDA
    if SESION is None:
        print("Primero ejecuta el paso 1.")
        return None
    semillas = [s.strip() for s in str(palabras_clave).split(",") if s.strip()]
    if not semillas:
        print("Escribe al menos una palabra clave en «palabras_clave».")
        return None
    lista_paises = []
    for nombre in str(paises).split(","):
        if nombre.strip():
            pais = buscar_pais(nombre)
            if pais and pais not in lista_paises:
                lista_paises.append(pais)
    if not lista_paises:
        print("Escribe al menos un país que exista, por ejemplo «España».")
        return None
    busquedas = len(semillas) * len(lista_paises)
    if busquedas > MAX_BUSQUEDAS:
        print(f"⚠️ Son {busquedas} búsquedas (palabras × países) y el tope es {MAX_BUSQUEDAS}.")
        print("Divídelo en varias tandas o sube MAX_BUSQUEDAS en el paso 1.")
        return None

    gasto_antes = GASTO["total"]
    codigo = IDIOMAS[idioma]
    filas = []
    idioma_usado = codigo
    for pais in lista_paises:
        cod = _idioma_del_pais(pais, codigo)
        if not cod:
            continue
        idioma_usado = cod
        for semilla in semillas:
            print(f"🔎 {semilla} · {_nombre(pais)}…")
            for kd in _ideas(semilla, pais, cod, profundidad):
                filas.append(_fila(kd, pais, semilla))
    if not filas:
        print("❌ No han salido ideas. Prueba con otra palabra, menos concreta, o con más profundidad.")
        print(f"Coste: {dolares(GASTO['total'] - gasto_antes)}")
        return None

    df = pd.DataFrame(filas)
    df["Volumen"] = pd.to_numeric(df["Volumen"], errors="coerce")
    for columna in df.columns:
        if columna.startswith(("Vol ", "Tendencia")) or columna == "Dificultad":
            df[columna] = pd.to_numeric(df[columna], errors="coerce").round().astype("Int64")
    # Si la misma búsqueda sale con dato y sin dato, se queda la que tiene dato.
    df = df.sort_values("Volumen", ascending=False, na_position="last").drop_duplicates(["País", "Palabra clave"])
    sin_dato = df["Volumen"].isna()
    df["Volumen"] = df["Volumen"].fillna(0).astype(int)
    avisos = []
    apartadas = []

    df = _apartar(df, sin_dato, SIN_DATO, apartadas)
    avisos.append(f"{int(sin_dato.sum())} de las que Google no da el volumen")

    ciudad = str(ciudad).strip()
    minimo = int(volumen_minimo) if volumen_minimo and int(volumen_minimo) > 0 else (50 if ciudad else 1)
    pocas = df["Volumen"] < minimo
    df = _apartar(df, pocas, "Sin búsquedas" if minimo == 1 else f"Menos de {minimo} búsquedas al mes", apartadas)
    avisos.append(f"{int(pocas.sum())} sin búsquedas" if minimo == 1 else f"{int(pocas.sum())} con menos de {minimo} búsquedas al mes")

    OTRAS_CIUDADES = None
    if ciudad:
        mias = _variantes(ciudad)
        en_semillas = {t for s in semillas for t in _variantes(s)} | {w for s in semillas for w in normalizar(s).split()}
        otras = _ciudades(lista_paises) - mias - en_semillas
        mencion = df["Palabra clave"].map(lambda k: _menciona(normalizar(k), otras))
        OTRAS_CIUDADES = df[mencion.notna()].assign(**{"Otra ciudad": mencion[mencion.notna()]})
        df = df[mencion.isna()]
        if len(OTRAS_CIUDADES):
            avisos.append(f"{len(OTRAS_CIUDADES)} que nombran otra ciudad (las tienes en la descarga, en «Otras ciudades»)")
    if quitar_sueltas:
        sueltas = df["Palabra clave"].str.split().str.len() <= 1
        df = _apartar(df, sueltas, "Una sola palabra", apartadas)
        avisos.append(f"{int(sueltas.sum())} de una sola palabra")

    if df.empty:
        print("❌ No queda ninguna idea. Estas son las que he quitado y por qué:")
        tabla = _tabla_descartadas(apartadas)
        if tabla is not None:
            mostrar(tabla, 25)
        print("Si es por los filtros, prueba con un volumen mínimo más bajo o sin quitar las de una sola palabra.")
        print("Si Google no da el volumen, no es que nadie lo busque: pasa en temas que Google no deja anunciar.")
        print(f"Coste de esta búsqueda: {dolares(GASTO['total'] - gasto_antes)}")
        return None

    df = _intenciones(df.copy(), idioma_usado)
    df = _ordenar(agrupar(df, idioma_usado, _lugares(lista_paises)))
    RESULTADOS = df
    FILTRADO = None
    DESCARTADAS = _tabla_descartadas(apartadas)
    BUSQUEDA = {"semillas": semillas, "paises": [_nombre(p) for p in lista_paises], "fecha": date.today()}

    grupos = resumen_grupos(df)
    coste = GASTO["total"] - gasto_antes
    saldo = _saldo()
    print()
    print(f"✅ {df['Palabra clave'].nunique()} palabras clave en {len(grupos)} grupos.")
    quitadas = [a for a in avisos if not a.startswith("0 ")]
    print("He quitado " + "; ".join(quitadas) + "." if quitadas else "No he quitado ninguna.")
    if DESCARTADAS is not None:
        print("No se pierden: las tienes en la descarga, en la hoja «Descartadas», con el motivo.")
    if sin_dato.any():
        print("Que Google no dé el volumen no quiere decir que nadie lo busque: pasa en temas que Google no deja anunciar.")
    print(f"Coste de esta búsqueda: {dolares(coste)}" + (f" · Saldo que te queda: {dolares(saldo)}" if saldo is not None else ""))
    print("\nGrupos, de más a menos búsquedas:")
    mostrar(grupos, 15)
    print("\nTodas las palabras clave:")
    mostrar(_columnas_vista(df), 25)
    return df


def _numero(texto):
    texto = str(texto).strip().replace(",", ".")
    if not texto:
        return None
    try:
        return float(texto)
    except ValueError:
        print(f"⚠️ «{texto}» no es un número; no lo tengo en cuenta.")
        return None


def _contiene(serie, terminos):
    normal = serie.map(normalizar)
    marca = pd.Series(False, index=serie.index)
    for termino in terminos:
        marca |= normal.str.contains(re.escape(normalizar(termino)), regex=True)
    return marca


def filtrar(incluir="", excluir="", volumen_min="", volumen_max="", dificultad_min="", dificultad_max="",
            cpc_min="", cpc_max="", intencion="Todas", grupo=""):
    global FILTRADO
    if RESULTADOS is None:
        print("Primero haz una búsqueda en el paso 2.")
        return None
    df = RESULTADOS
    incluir = [t.strip() for t in str(incluir).split(",") if t.strip()]
    excluir = [t.strip() for t in str(excluir).split(",") if t.strip()]
    if incluir:
        df = df[_contiene(df["Palabra clave"], incluir)]
    if excluir:
        df = df[~_contiene(df["Palabra clave"], excluir)]
    for columna, minimo, maximo in (("Volumen", volumen_min, volumen_max),
                                    ("Dificultad", dificultad_min, dificultad_max),
                                    ("CPC", cpc_min, cpc_max)):
        valores = pd.to_numeric(df[columna], errors="coerce")
        minimo, maximo = _numero(minimo), _numero(maximo)
        if minimo is not None:
            df, valores = df[valores >= minimo], valores[valores >= minimo]
        if maximo is not None:
            df = df[valores <= maximo]
    if intencion and intencion != "Todas":
        df = df[df["Intención"] == intencion]
    if str(grupo).strip():
        df = df[_contiene(df["Grupo"], [grupo])]
    FILTRADO = df.reset_index(drop=True)
    print(f"✅ Quedan {FILTRADO['Palabra clave'].nunique()} de {RESULTADOS['Palabra clave'].nunique()} palabras clave.")
    if FILTRADO.empty:
        print("Ningún resultado: afloja algún filtro. Tu búsqueda sigue entera; filtrar no la borra.")
        return FILTRADO
    print("\nGrupos:")
    mostrar(resumen_grupos(FILTRADO), 15)
    print("\nPalabras clave:")
    mostrar(_columnas_vista(FILTRADO), 25)
    return FILTRADO


def _ajustar_hoja(hoja):
    hoja.freeze_panes = "A2"
    for columna in hoja.columns:
        ancho = max(len(str(c.value)) if c.value is not None else 0 for c in columna[:200])
        hoja.column_dimensions[columna[0].column_letter].width = min(max(ancho + 2, 8), 60)


def descargar(formato="Excel (.xlsx)", que="Lo filtrado"):
    if RESULTADOS is None:
        print("Primero haz una búsqueda en el paso 2.")
        return None
    df = RESULTADOS
    if que == "Lo filtrado":
        if FILTRADO is None:
            print("No has filtrado nada: descargo la búsqueda entera.")
        else:
            df = FILTRADO
    if df.empty:
        print("No hay nada que descargar con estos filtros.")
        return None
    nombre = re.sub(r"[^a-z0-9]+", "-", normalizar(BUSQUEDA["semillas"][0])).strip("-") or "busqueda"
    ruta = f"palabras-clave_{nombre}_{BUSQUEDA['fecha']:%Y-%m-%d}"
    if formato.startswith("Excel"):
        ruta += ".xlsx"
        with pd.ExcelWriter(ruta, engine="openpyxl") as libro:
            df.to_excel(libro, sheet_name="Palabras clave", index=False)
            resumen_grupos(df).to_excel(libro, sheet_name="Grupos", index=False)
            if DESCARTADAS is not None and len(DESCARTADAS):
                DESCARTADAS.to_excel(libro, sheet_name="Descartadas", index=False)
            if OTRAS_CIUDADES is not None and len(OTRAS_CIUDADES):
                OTRAS_CIUDADES.to_excel(libro, sheet_name="Otras ciudades", index=False)
            for hoja in libro.sheets.values():
                _ajustar_hoja(hoja)
    else:
        ruta += ".csv"
        df.to_csv(ruta, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print(f"✅ {ruta}: {df['Palabra clave'].nunique()} palabras clave.")
    if EN_COLAB:
        files.download(ruta)
    return ruta


conectar()


In [ ]:
#@title 2. Busca ideas de palabras clave { display-mode: "form" }
#@markdown Una o varias palabras clave, separadas por comas.
palabras_clave = "" #@param {type:"string"}
#@markdown Uno o varios países, separados por comas (por ejemplo: España, México).
paises = "España" #@param {type:"string"}
idioma = "Español" #@param ["Español", "Catalán", "Inglés", "Francés", "Alemán", "Italiano", "Portugués", "Neerlandés", "Sueco", "Ruso", "Turco", "Árabe", "Hindi", "Japonés", "Coreano", "Chino (simplificado)"]
#@markdown **Profundidad:** 1 trae pocas ideas y muy pegadas a tu palabra; 2, el equilibrio; 3, muchas más, pero se aleja más del tema y cuesta algo más.
profundidad = 2 #@param {type:"slider", min:1, max:3, step:1}
#@markdown **Ciudad:** si la búsqueda es local (por ejemplo, «abogado lleida»), escribe aquí la ciudad y se apartan las búsquedas que nombran otras ciudades. Si no, déjalo vacío.
ciudad = "" #@param {type:"string"}
quitar_las_de_una_sola_palabra = True #@param {type:"boolean"}
#@markdown **Volumen mínimo** de búsquedas al mes. Con 0 se elige solo: 1, o 50 si has puesto ciudad.
volumen_minimo = 0 #@param {type:"integer"}

if "buscar" not in globals():
    print("Primero ejecuta el paso 1.")
else:
    buscar(palabras_clave, paises, idioma, profundidad, ciudad, quitar_las_de_una_sola_palabra, volumen_minimo)


In [ ]:
#@title 3. Filtra los resultados (no gasta saldo) { display-mode: "form" }
#@markdown Cambia lo que quieras y vuelve a ejecutar: cada vez filtra sobre la búsqueda entera, no sobre el filtro anterior. Deja vacío lo que no quieras usar.
#@markdown
#@markdown Solo las que contengan alguna de estas palabras (separadas por comas):
incluir = "" #@param {type:"string"}
#@markdown Quitar las que contengan alguna de estas palabras:
excluir = "" #@param {type:"string"}
volumen_min = "" #@param {type:"string"}
volumen_max = "" #@param {type:"string"}
dificultad_min = "" #@param {type:"string"}
dificultad_max = "" #@param {type:"string"}
cpc_min = "" #@param {type:"string"}
cpc_max = "" #@param {type:"string"}
intencion = "Todas" #@param ["Todas", "Informativa", "Comercial", "Transaccional", "Navegacional"]
#@markdown Solo un grupo (escribe parte de su nombre):
grupo = "" #@param {type:"string"}

if "filtrar" not in globals():
    print("Primero ejecuta el paso 1.")
else:
    filtrar(incluir, excluir, volumen_min, volumen_max, dificultad_min, dificultad_max, cpc_min, cpc_max, intencion, grupo)


In [ ]:
#@title 4. Descarga { display-mode: "form" }
#@markdown El Excel trae las palabras clave (con el volumen de cada mes), el resumen de los grupos, las descartadas con su motivo (Google no da el volumen, sin búsquedas o de una sola palabra) y, si pusiste ciudad, las que nombran otras ciudades.
formato = "Excel (.xlsx)" #@param ["Excel (.xlsx)", "CSV (separado por ;)"]
que_descargar = "Lo filtrado" #@param ["Lo filtrado", "Todo"]

if "descargar" not in globals():
    print("Primero ejecuta el paso 1.")
else:
    descargar(formato, que_descargar)


---

**¿Y ahora qué búsquedas trabajo?** Esta tabla te dice qué busca la gente. Cuáles te conviene trabajar depende de tu negocio, de lo que ya tiene tu web y de quién se lleva hoy esas búsquedas.

**¿Lo quieres adaptado a tu empresa?** Si necesitas decidir qué búsquedas trabajar en tu web, o una herramienta pensada para tu caso, cuéntamelo en [dupavi.es](https://dupavi.es/recursos/planificador-palabras-clave/#formulario).
